# Scenario Testing v2.1 (Fast Parallel Map Build)

This notebook is a single, self-contained workflow to:
1. run scenario comparisons in parallel over the largest wards,
2. standardize outputs to **kWh/year per household**,
3. generate stakeholder-ready summary plots,
4. export one interactive HTML map with scenario toggles.

## Runtime design
- Default mode is **fast annualization**: simulate a short window and scale to annual equivalent.
- This is intentional: true 8,760-hour runs across multiple scenarios are usually too slow for a ~5 minute notebook run.
- Set `USE_FAST_ANNUALIZATION = False` if you need exact annual simulation and accept longer runtime.

In [ ]:
# Imports
from __future__ import annotations

from pathlib import Path
import multiprocessing as mp
import tempfile
import random
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
import branca.colormap as cm
import matplotlib.pyplot as plt
import yaml

from household_energy.model import EnergyModel

warnings.filterwarnings('ignore', message='.*GeoSeries.notna.*')

## 1) User settings
Tune only this cell for most runs.

In [ ]:
# Paths
GEOJSON = Path('../data/epc_abm_newcastle.geojson')
CLIMATE = Path('../data/ncc_2t_timeseries_2010_2039.parquet')
HIDP_CSV = Path('../data/hidp_uprn_matches_tiered.csv')
OUTDIR = Path('results/scenario_v2_1')
OUTDIR.mkdir(parents=True, exist_ok=True)

# Scope + runtime
AREA_COLUMN = 'ward_code'              # reporting/filter geography in outputs
AREA_SCOPE = 'top_n'                   # 'top_n' or 'city'
TOP_AREAS_N = 4                        # used when AREA_SCOPE='top_n'
PROCESS_COLUMN_PREFERRED = 'lsoa_code' # parallel shard unit; falls back to AREA_COLUMN
N_PROCS = max(1, min(8, (mp.cpu_count() or 2) - 1))

USE_FAST_ANNUALIZATION = False         # False = true annual run
WINDOW_DAYS = 21                       # used if fast mode is True
RUN_YEARS_EXACT = 1                    # used if fast mode is False
START_UTC = '2020-01-01T00:00:00Z'

# Map rendering guardrail
MAX_POINTS_PER_LAYER = 6000

# Outputs
MAP_OUT_HTML = OUTDIR / 'scenario_comparison_map.html'
LONG_OUT_PARQUET = OUTDIR / 'scenario_map_long.parquet'
LONG_OUT_CSV = OUTDIR / 'scenario_map_long.csv'
SUMMARY_OUT_CSV = OUTDIR / 'scenario_summary.csv'

WINDOW_HOURS = (WINDOW_DAYS * 24) if USE_FAST_ANNUALIZATION else (RUN_YEARS_EXACT * 365 * 24)
ANNUAL_FACTOR = (365.0 / WINDOW_DAYS) if USE_FAST_ANNUALIZATION else 1.0

print(f'Area column: {AREA_COLUMN}')
print(f'Area scope: {AREA_SCOPE}')
print(f'Top areas (if top_n): {TOP_AREAS_N}')
print(f'Preferred process column: {PROCESS_COLUMN_PREFERRED}')
print(f'Parallel workers: {N_PROCS}')
print(f'Window hours: {WINDOW_HOURS:,}')
print(f'Annualization factor: {ANNUAL_FACTOR:.3f}')
print(f'Outputs -> {OUTDIR.resolve()}')

## 2) Load + enrich + select largest areas

In [ ]:
def load_enriched_gdf(geojson_path: Path, hidp_csv_path: Path | None) -> gpd.GeoDataFrame:
    g = gpd.read_file(geojson_path)
    g['UPRN'] = g['UPRN'].astype(str).str.strip()

    if hidp_csv_path and hidp_csv_path.exists():
        hidp = pd.read_csv(hidp_csv_path, low_memory=False)
        hidp.columns = [c.strip() for c in hidp.columns]
        hidp['uprn_chr'] = hidp['uprn_chr'].astype(str).str.strip()
        hidp = hidp.drop_duplicates(subset=['uprn_chr'])
        g = g.merge(hidp, how='left', left_on='UPRN', right_on='uprn_chr', suffixes=('_geo', '_hidp'))

        for base in ['lsoa_code', 'ward_code', 'local_authority']:
            geo_col, hidp_col = f'{base}_geo', f'{base}_hidp'
            if base not in g.columns and (geo_col in g.columns or hidp_col in g.columns):
                if geo_col in g.columns and hidp_col in g.columns:
                    g[base] = g[geo_col].combine_first(g[hidp_col])
                elif geo_col in g.columns:
                    g[base] = g[geo_col]
                else:
                    g[base] = g[hidp_col]

    return g


def to_wgs84_points(gdf_in: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    g = gdf_in.dropna(subset=['geometry']).copy()
    g = g.to_crs(4326) if g.crs else g.set_crs(4326)
    if (g.geometry.geom_type != 'Point').any():
        g['geometry'] = g.geometry.centroid
    return g


gdf_all = load_enriched_gdf(GEOJSON, HIDP_CSV if HIDP_CSV.exists() else None)
if AREA_COLUMN not in gdf_all.columns:
    raise KeyError(f'{AREA_COLUMN} not found. Available columns include: {list(gdf_all.columns)[:20]}')

all_area_counts = gdf_all[AREA_COLUMN].astype(str).replace('', np.nan).dropna().value_counts()
if AREA_SCOPE == 'city':
    selected_areas = all_area_counts.index.tolist()
    area_counts = all_area_counts
else:
    area_counts = all_area_counts.head(TOP_AREAS_N)
    selected_areas = area_counts.index.tolist()

print('Selected areas:')
print(area_counts.head(TOP_AREAS_N if AREA_SCOPE != 'city' else 12))

# Focus data for selected area scope
gdf_focus = gdf_all[gdf_all[AREA_COLUMN].astype(str).isin(selected_areas)].copy()
gdf_focus = to_wgs84_points(gdf_focus)
gdf_focus['AgentID'] = gdf_focus['UPRN'].astype(str)

# Process shards (LSOA preferred for parallel speed and balancing).
PROCESS_COLUMN = PROCESS_COLUMN_PREFERRED if PROCESS_COLUMN_PREFERRED in gdf_focus.columns else AREA_COLUMN
if PROCESS_COLUMN != AREA_COLUMN:
    selected_units = (
        gdf_focus[PROCESS_COLUMN].astype(str).replace('', np.nan).dropna().unique().tolist()
    )
else:
    selected_units = selected_areas.copy()

print(f'Focus rows: {len(gdf_focus):,}')
print(f'Process column: {PROCESS_COLUMN} | units: {len(selected_units):,}')

## 3) Scenario definitions (single source of truth)

In [ ]:
SCENARIOS = [
    'hp_low_income_education',
    'hp_social_rent',
    'hp_top_users',
    'retrofit_kids',
    'retrofit_elderly',
]

META_COLS = [
    'AgentID', 'UPRN', 'geometry', AREA_COLUMN,
    'property_type', 'tenure', 'hh_income_band', 'hh_edu_detail', 'schedule_type', 'hh_children'
]
for col in META_COLS:
    if col != 'geometry' and col not in gdf_focus.columns:
        gdf_focus[col] = np.nan

# Top-users cohort from baseline proxy: calibrated annual energy if present; else floor area proxy.
if 'energy_cal_kwh' in gdf_focus.columns:
    ranking = pd.to_numeric(gdf_focus['energy_cal_kwh'], errors='coerce').fillna(0.0)
else:
    ranking = pd.to_numeric(gdf_focus.get('floor_area_m2', 0), errors='coerce').fillna(0.0)

cut = max(int(len(gdf_focus) * 0.10), 1)
top_users_ids = set(gdf_focus.loc[ranking.nlargest(cut).index, 'AgentID'].astype(str))

sched = gdf_focus.get('schedule_type', pd.Series('', index=gdf_focus.index)).astype(str).str.lower()

MASKS = {
    'hp_low_income_education': (
        gdf_focus.get('hh_income_band', pd.Series('', index=gdf_focus.index)).isin(['q1_lowest', 'q2_low'])
        & gdf_focus.get('hh_edu_detail', pd.Series('', index=gdf_focus.index)).isin(['education_other', 'upper_further'])
    ),
    'hp_social_rent': gdf_focus.get('tenure', pd.Series('', index=gdf_focus.index)).astype(str).str.lower().eq('social_rent'),
    'hp_top_users': gdf_focus['AgentID'].isin(top_users_ids),
    'retrofit_kids': (
        gdf_focus.get('hh_children', pd.Series(False, index=gdf_focus.index)).fillna(False).astype(bool)
        | sched.isin(['family_with_children', 'single_parent_with_children'])
    ),
    'retrofit_elderly': sched.eq('retired_household'),
}

cohort_sizes = {k: int(v.fillna(False).sum()) for k, v in MASKS.items()}
pd.Series(cohort_sizes, name='cohort_size').to_frame()

## 4) Parallel model runner (area batch pattern)

In [ ]:
UNIT_INPUT_DIR = OUTDIR / 'unit_inputs'
UNIT_INPUT_DIR.mkdir(parents=True, exist_ok=True)

for unit_val in selected_units:
    g_unit = gdf_focus[gdf_focus[PROCESS_COLUMN].astype(str) == str(unit_val)].copy()
    g_unit.to_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet', index=False)


def _seed_from(*parts) -> int:
    return abs(hash('|'.join(map(str, parts)))) % (2**32)


def _run_model_household_window(gdf_in: gpd.GeoDataFrame, *, start_utc: str, hours: int, cfg: dict | None) -> pd.DataFrame:
    cfg_path = None
    if cfg is not None:
        with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as tmp:
            yaml.safe_dump(cfg, tmp)
            cfg_path = tmp.name

    # Annual extraction path: no agent DataCollector to reduce memory/runtime.
    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(CLIMATE),
        climate_start=start_utc,
        collect_agent_level=False,
        agent_collect_every=168,
        config_path=cfg_path,
    )

    for _ in range(int(hours)):
        m.step()

    rows = []
    for h in m.household_agents:
        aid = str(getattr(h, 'unique_id', ''))
        by_year = getattr(h, 'annual_kwh_by_year', {}) or {}
        kwh_window = float(sum(by_year.values()))
        rows.append({'AgentID': aid, 'kwh_window': kwh_window})

    out = pd.DataFrame(rows)
    if out.empty:
        out = pd.DataFrame({'AgentID': gdf_in['UPRN'].astype(str), 'kwh_window': 0.0})
    out['AgentID'] = out['AgentID'].astype(str)
    return out


def _apply_policy(gdf_unit: gpd.GeoDataFrame, scenario: str, in_cohort: pd.Series) -> tuple[gpd.GeoDataFrame, dict | None]:
    gp = gdf_unit.copy()
    mask = in_cohort.fillna(False).astype(bool)

    if scenario.startswith('hp_'):
        gp['is_heatpump_candidate'] = 0
        gp.loc[mask, 'is_heatpump_candidate'] = 1
        gp.loc[mask, 'heatpump_candidate_class'] = 'priority'
        cfg = {'meta': {'name': scenario}, 'model': {'heatpump_adoption_rate': 1.0}}
    elif scenario == 'retrofit_kids':
        gp['retrofit_envelope_score'] = gp.get('retrofit_envelope_score', 0.5)
        gp['retrofit_envelope_score'] = pd.to_numeric(gp['retrofit_envelope_score'], errors='coerce').fillna(0.5)
        gp.loc[mask, 'retrofit_envelope_score'] = 1.0
        cfg = {'meta': {'name': scenario}, 'model': {'heatpump_adoption_rate': 0.0}}
    elif scenario == 'retrofit_elderly':
        gp['retrofit_envelope_score'] = gp.get('retrofit_envelope_score', 0.5)
        gp['retrofit_envelope_score'] = pd.to_numeric(gp['retrofit_envelope_score'], errors='coerce').fillna(0.5)
        gp.loc[mask, 'retrofit_envelope_score'] = (gp.loc[mask, 'retrofit_envelope_score'] + 0.2).clip(0, 1)
        cfg = {'meta': {'name': scenario}, 'model': {'heatpump_adoption_rate': 0.0}}
    else:
        raise ValueError(f'Unknown scenario: {scenario}')

    return gp, cfg


def _run_unit_baseline(unit_val: str) -> pd.DataFrame:
    g = gpd.read_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet')
    g['AgentID'] = g['UPRN'].astype(str)

    seed = _seed_from('baseline', unit_val)
    np.random.seed(seed)
    random.seed(seed)

    cfg = {'meta': {'name': 'baseline'}, 'model': {'heatpump_adoption_rate': 0.0}}
    res = _run_model_household_window(g, start_utc=START_UTC, hours=WINDOW_HOURS, cfg=cfg)

    keep = [c for c in META_COLS if c in g.columns]
    meta = g[keep].copy()
    meta['AgentID'] = meta['AgentID'].astype(str)

    out = meta.merge(res.rename(columns={'kwh_window': 'baseline_kwh_window'}), on='AgentID', how='left')
    out[PROCESS_COLUMN] = str(unit_val)
    return out


def _run_unit_scenario(unit_val: str, scenario: str) -> pd.DataFrame:
    g = gpd.read_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet')
    g['AgentID'] = g['UPRN'].astype(str)

    seed = _seed_from('policy', unit_val, scenario)
    np.random.seed(seed)
    random.seed(seed)

    mask_all = MASKS[scenario]
    m_unit = gdf_focus[PROCESS_COLUMN].astype(str).eq(str(unit_val))
    cohort_ids = set(gdf_focus.loc[m_unit & mask_all.fillna(False), 'AgentID'].astype(str))
    in_cohort = g['AgentID'].isin(cohort_ids)

    gp, cfg = _apply_policy(g, scenario, in_cohort)
    res = _run_model_household_window(gp, start_utc=START_UTC, hours=WINDOW_HOURS, cfg=cfg)

    cohort_df = g[['AgentID']].copy()
    cohort_df['in_cohort'] = in_cohort.values

    out = cohort_df.merge(res.rename(columns={'kwh_window': 'policy_kwh_window'}), on='AgentID', how='left')
    out['AgentID'] = out['AgentID'].astype(str)
    out['scenario'] = scenario
    out[PROCESS_COLUMN] = str(unit_val)
    return out


def _run_parallel_starmap(func, tasks, n_procs: int):
    if n_procs <= 1:
        return [func(*t) if isinstance(t, tuple) else func(t) for t in tasks]

    # Same multiprocessing pattern as run_lsoa_batch (fork -> pool -> starmap/map).
    try:
        ctx = mp.get_context('fork')
        with ctx.Pool(processes=n_procs) as pool:
            if tasks and isinstance(tasks[0], tuple):
                return pool.starmap(func, tasks)
            return pool.map(func, tasks)
    except Exception as e:
        print(f'Parallel pool failed ({e}); falling back to serial.')
        return [func(*t) if isinstance(t, tuple) else func(t) for t in tasks]

## 5) Run baseline + scenarios (parallel) and build annualized long table

In [ ]:
baseline_tasks = [str(u) for u in selected_units]
scenario_tasks = [(str(u), s) for u in selected_units for s in SCENARIOS]

print(f'Baseline tasks: {len(baseline_tasks)}')
print(f'Policy tasks: {len(scenario_tasks)}')
print(f'Process column: {PROCESS_COLUMN}')

baseline_rows = _run_parallel_starmap(_run_unit_baseline, baseline_tasks, n_procs=N_PROCS)
baseline_df = pd.concat(baseline_rows, ignore_index=True)

policy_rows = _run_parallel_starmap(_run_unit_scenario, scenario_tasks, n_procs=N_PROCS)
policy_df = pd.concat(policy_rows, ignore_index=True)

scenario_long = (
    policy_df
    .merge(
        baseline_df[['AgentID', 'UPRN', AREA_COLUMN, 'geometry', 'baseline_kwh_window', 'property_type', 'tenure', 'hh_income_band', 'hh_edu_detail', 'schedule_type', 'hh_children']],
        on='AgentID',
        how='left',
    )
)

# Keep geometry as a GeoDataFrame in-memory for map rendering.
scenario_long = gpd.GeoDataFrame(scenario_long, geometry='geometry', crs=gdf_focus.crs)

scenario_long['baseline_kwh_year'] = scenario_long['baseline_kwh_window'] * ANNUAL_FACTOR
scenario_long['policy_kwh_year'] = scenario_long['policy_kwh_window'] * ANNUAL_FACTOR
scenario_long['delta_kwh_year'] = scenario_long['policy_kwh_year'] - scenario_long['baseline_kwh_year']

# Persist flat outputs only (geometry stays in-memory for mapping)
scenario_long.drop(columns=['geometry'], errors='ignore').to_parquet(LONG_OUT_PARQUET, index=False)
scenario_long.drop(columns=['geometry'], errors='ignore').to_csv(LONG_OUT_CSV, index=False)

summary = (
    scenario_long.groupby('scenario', as_index=False)
    .agg(
        n_households=('AgentID', 'nunique'),
        cohort_size=('in_cohort', 'sum'),
        mean_base_kwh_year=('baseline_kwh_year', 'mean'),
        mean_policy_kwh_year=('policy_kwh_year', 'mean'),
        mean_delta_kwh_year=('delta_kwh_year', 'mean'),
        median_delta_kwh_year=('delta_kwh_year', 'median'),
    )
    .sort_values('mean_delta_kwh_year')
)
summary.to_csv(SUMMARY_OUT_CSV, index=False)

print(f'Saved flat parquet: {LONG_OUT_PARQUET}')
print(f'Saved csv: {LONG_OUT_CSV}')
print(f'Saved: {SUMMARY_OUT_CSV}')
summary

## 6) Scenario visuals for stakeholders

In [ ]:
plot_df = summary.copy()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(plot_df['scenario'], plot_df['mean_delta_kwh_year'], color='#2a6f97')
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('Mean delta (kWh/year per household)')
ax.set_title('Scenario impact summary (annualized)')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

# Why prior plots looked flat:
# most households are untreated in each scenario, so delta ~ 0 piles up and hides the cohort signal.
# Show both all-household and cohort-focused diagnostics.
viz = scenario_long[['scenario', 'delta_kwh_year', 'in_cohort']].dropna().copy()
if len(viz) > 120000:
    viz = viz.sample(120000, random_state=42)

scenarios = summary['scenario'].tolist()
rows = []
for s in scenarios:
    d = viz[viz['scenario'] == s]
    all_v = d['delta_kwh_year'].astype(float)
    coh_v = d.loc[d['in_cohort'].fillna(False).astype(bool), 'delta_kwh_year'].astype(float)
    non_v = d.loc[~d['in_cohort'].fillna(False).astype(bool), 'delta_kwh_year'].astype(float)

    rows.append({
        'scenario': s,
        'n_all': int(len(all_v)),
        'n_cohort': int(len(coh_v)),
        'mean_all': float(all_v.mean()) if len(all_v) else np.nan,
        'median_all': float(all_v.median()) if len(all_v) else np.nan,
        'mean_cohort': float(coh_v.mean()) if len(coh_v) else np.nan,
        'median_cohort': float(coh_v.median()) if len(coh_v) else np.nan,
        'p10_cohort': float(np.nanpercentile(coh_v, 10)) if len(coh_v) else np.nan,
        'p90_cohort': float(np.nanpercentile(coh_v, 90)) if len(coh_v) else np.nan,
        'pct_all_near_zero(|Δ|<1)': float((all_v.abs() < 1.0).mean() * 100.0) if len(all_v) else np.nan,
        'pct_cohort_near_zero(|Δ|<1)': float((coh_v.abs() < 1.0).mean() * 100.0) if len(coh_v) else np.nan,
    })

scenario_stats = pd.DataFrame(rows).sort_values('median_cohort')
scenario_stats

fig, axes = plt.subplots(len(scenarios), 2, figsize=(12, 2.7 * len(scenarios)))
if len(scenarios) == 1:
    axes = np.array([axes])

for i, s in enumerate(scenarios):
    d = viz[viz['scenario'] == s]
    all_v = d['delta_kwh_year'].astype(float)
    coh_v = d.loc[d['in_cohort'].fillna(False).astype(bool), 'delta_kwh_year'].astype(float)

    # Scenario-specific zoom from cohort distribution keeps signal visible.
    if len(coh_v) >= 20:
        lo, hi = np.nanpercentile(coh_v, [1, 99])
    elif len(all_v) > 0:
        lo, hi = np.nanpercentile(all_v, [1, 99])
    else:
        lo, hi = -1, 1
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        lo, hi = float(all_v.min() if len(all_v) else -1), float(all_v.max() if len(all_v) else 1)
        if lo == hi:
            lo, hi = lo - 1, hi + 1

    # Left: cohort-only histogram (log y) so tails aren't invisible.
    axl = axes[i, 0]
    if len(coh_v):
        clip_coh = coh_v.clip(lo, hi)
        axl.hist(clip_coh, bins=35, color='#5f9e6e', alpha=0.9)
        axl.set_yscale('log')
    axl.axvline(0, color='black', linewidth=1)
    axl.set_xlim(lo, hi)
    axl.set_title(f'{s} — cohort distribution')
    axl.set_ylabel('count (log)')

    # Right: ECDF all vs cohort (same x-range) to compare impact concentration.
    axr = axes[i, 1]
    if len(all_v):
        x_all = np.sort(all_v.clip(lo, hi).to_numpy())
        y_all = np.arange(1, len(x_all) + 1) / len(x_all)
        axr.plot(x_all, y_all, label='all households', color='#4c78a8', linewidth=1.7)
    if len(coh_v):
        x_coh = np.sort(coh_v.clip(lo, hi).to_numpy())
        y_coh = np.arange(1, len(x_coh) + 1) / len(x_coh)
        axr.plot(x_coh, y_coh, label='cohort only', color='#f58518', linewidth=1.7)
    axr.axvline(0, color='black', linewidth=1)
    axr.set_xlim(lo, hi)
    axr.set_ylim(0, 1)
    axr.set_title(f'{s} — ECDF comparison')
    axr.set_ylabel('cumulative share')
    axr.legend(loc='lower right', frameon=False)

axes[-1, 0].set_xlabel('Delta kWh/year (scenario p1..p99 zoom)')
axes[-1, 1].set_xlabel('Delta kWh/year (scenario p1..p99 zoom)')
plt.tight_layout()
plt.show()

## 7) Build exportable map (scenario + mode toggles + macro best/worst)

In [ ]:
# Ensure geometry is available even if scenario_long was reloaded as a flat DataFrame.
if 'geometry' not in scenario_long.columns:
    geom_lookup = gdf_focus[['AgentID', 'geometry']].drop_duplicates('AgentID').copy()
    scenario_long = scenario_long.merge(geom_lookup, on='AgentID', how='left')

scenario_long = gpd.GeoDataFrame(scenario_long, geometry='geometry', crs=gdf_focus.crs)
scenario_long = scenario_long.dropna(subset=['geometry']).copy()

vals = pd.concat([
    scenario_long['baseline_kwh_year'],
    scenario_long['delta_kwh_year'],
], axis=0).replace([np.inf, -np.inf], np.nan).dropna()
if vals.empty:
    raise ValueError('No values available for mapping.')

scale_max = float(np.nanpercentile(np.abs(vals), 99))
if not np.isfinite(scale_max) or scale_max <= 0:
    scale_max = 1.0

cmap = cm.LinearColormap(
    colors=['#00d4ff', '#f7f7f7', '#ff4d4f'],
    vmin=-scale_max,
    vmax=scale_max,
)
cmap.caption = f'Shared annual kWh scale: {-scale_max:,.0f} to {scale_max:,.0f}'

center = [float(scenario_long.geometry.y.mean()), float(scenario_long.geometry.x.mean())]
fmap = folium.Map(location=center, zoom_start=12, tiles='CartoDB positron')

header = f"""
<div style="position: fixed; top: 10px; left: 50px; z-index: 9999;
     background: rgba(255,255,255,.92); padding: 8px 12px; border-radius: 6px;
     font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; font-size: 13px;">
  <b>Scenario Comparison (annualized kWh/year)</b><br>
  Layers: baseline + delta | Selected areas: {TOP_AREAS_N} | UPRN hidden
</div>
"""
fmap.get_root().html.add_child(folium.Element(header))


def _clip(v):
    if pd.isna(v):
        return np.nan
    return float(np.clip(v, -scale_max, scale_max))


def _popup(r, mode):
    base = r.get('baseline_kwh_year', np.nan)
    pol = r.get('policy_kwh_year', np.nan)
    dlt = r.get('delta_kwh_year', np.nan)
    return folium.Popup(
        f"""
        <b>Scenario:</b> {r.get('scenario','')}<br>
        <b>Layer:</b> {mode}<br>
        <b>Baseline kWh/year:</b> {base:,.1f}<br>
        <b>Policy kWh/year:</b> {pol:,.1f}<br>
        <b>Delta (policy - baseline):</b> {dlt:,.1f}<br>
        <b>In cohort:</b> {bool(r.get('in_cohort', False))}<br>
        <hr style='margin:6px 0;'>
        <b>{AREA_COLUMN}:</b> {r.get(AREA_COLUMN,'')}<br>
        <b>Property:</b> {r.get('property_type','')}<br>
        <b>Tenure:</b> {r.get('tenure','')}<br>
        <b>Income:</b> {r.get('hh_income_band','')}<br>
        <b>Education:</b> {r.get('hh_edu_detail','')}<br>
        <b>Schedule:</b> {r.get('schedule_type','')}<br>
        <b>Children:</b> {r.get('hh_children','')}
        """,
        max_width=360,
    )

scenario_order = summary['scenario'].tolist()
layer_refs = {}

for s in scenario_order:
    sdf_all = scenario_long[scenario_long['scenario'] == s].copy()

    # baseline layer
    layer_base = folium.FeatureGroup(name=f'{s} • baseline', show=False)
    src_base = sdf_all if len(sdf_all) <= MAX_POINTS_PER_LAYER else sdf_all.sample(MAX_POINTS_PER_LAYER, random_state=42)
    for _, r in src_base.iterrows():
        v = _clip(r['baseline_kwh_year'])
        col = '#9e9e9e' if pd.isna(v) else cmap(v)
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3.0,
            color='#0b0f14',
            weight=0.6,
            fill=True,
            fill_color=col,
            fill_opacity=0.88,
            popup=_popup(r, 'baseline'),
        ).add_to(layer_base)
    layer_base.add_to(fmap)
    layer_refs[(s, 'baseline')] = layer_base.get_name()

    # delta layer
    sdf_delta = sdf_all[
        sdf_all['in_cohort'].fillna(False).astype(bool)
        & (sdf_all['delta_kwh_year'].fillna(0).abs() > 1e-9)
    ].copy()

    layer_delta = folium.FeatureGroup(name=f'{s} • delta', show=False)
    src_delta = sdf_delta if len(sdf_delta) <= MAX_POINTS_PER_LAYER else sdf_delta.sample(MAX_POINTS_PER_LAYER, random_state=42)
    for _, r in src_delta.iterrows():
        v = _clip(r['delta_kwh_year'])
        col = '#9e9e9e' if pd.isna(v) else cmap(v)
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3.2,
            color='#0b0f14',
            weight=0.7,
            fill=True,
            fill_color=col,
            fill_opacity=0.92,
            popup=_popup(r, 'delta'),
        ).add_to(layer_delta)
    layer_delta.add_to(fmap)
    layer_refs[(s, 'delta')] = layer_delta.get_name()

    # delta top20 layer
    if len(sdf_delta):
        thr = float(np.nanpercentile(sdf_delta['delta_kwh_year'].abs(), 80))
    else:
        thr = np.inf
    sdf_top = sdf_delta[sdf_delta['delta_kwh_year'].abs() >= thr].copy()

    layer_top = folium.FeatureGroup(name=f'{s} • delta_top20', show=False)
    src_top = sdf_top if len(sdf_top) <= MAX_POINTS_PER_LAYER else sdf_top.sample(MAX_POINTS_PER_LAYER, random_state=42)
    for _, r in src_top.iterrows():
        v = _clip(r['delta_kwh_year'])
        col = '#9e9e9e' if pd.isna(v) else cmap(v)
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3.8,
            color='#000000',
            weight=0.9,
            fill=True,
            fill_color=col,
            fill_opacity=0.96,
            popup=_popup(r, 'delta_top20'),
        ).add_to(layer_top)
    layer_top.add_to(fmap)
    layer_refs[(s, 'delta_top20')] = layer_top.get_name()

# Scenario guide panel
scenario_guide_html = f"""
<div style="position: fixed; top: 84px; left: 50px; z-index: 9998;
     width: 360px; max-height: 70vh; overflow-y: auto;
     background: rgba(255,255,255,.94); padding: 10px 12px; border-radius: 8px;
     box-shadow: 0 2px 10px rgba(0,0,0,.25);
     font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; font-size: 12px; line-height: 1.35;">
  <details>
    <summary style="cursor:pointer; font-weight:700;">Scenario Guide (click to expand)</summary>
    <div style="margin-top:8px;">
      <b>How to read this map</b><br>
      • <b>Baseline</b>: expected annual kWh with no intervention.<br>
      • <b>Delta</b>: policy - baseline annual kWh (negative means savings).<br>
      • Delta dots show only treated homes with non-zero impact.<br>
      • Values are annualized from the configured run window (exact annual if fast mode is off).
      <hr style='margin:7px 0;'>
      <b>Scenarios</b><br>
      • <b>hp_low_income_education</b>: heat-pump targeting for lower income + selected education groups.<br>
      • <b>hp_social_rent</b>: heat-pump targeting for social-rent households.<br>
      • <b>hp_top_users</b>: heat-pump targeting for top baseline energy users.<br>
      • <b>retrofit_kids</b>: stronger envelope retrofit for households with children profiles.<br>
      • <b>retrofit_elderly</b>: moderate envelope retrofit for retired-household profiles.
    </div>
  </details>
</div>
"""
fmap.get_root().html.add_child(folium.Element(scenario_guide_html))

# Custom controls (replaces Leaflet layer checklist)
opts = ''.join([f'<option value="{s}">{s}</option>' for s in scenario_order])
controls_html = f"""
<div style="position: fixed; top: 84px; right: 14px; z-index: 9998;
     width: 280px; background: rgba(255,255,255,.94); padding: 10px 12px;
     border-radius: 8px; box-shadow: 0 2px 10px rgba(0,0,0,.25);
     font-family: system-ui, -apple-system, Segoe UI, Roboto, sans-serif; font-size: 12px; line-height: 1.35;">
  <b>View Controls</b><br>
  <label for="scenarioSelect" style="display:block; margin-top:6px;">Scenario</label>
  <select id="scenarioSelect" style="width:100%; padding:4px;">{opts}</select>
  <label style="display:flex; align-items:center; gap:8px; margin-top:8px;">
    <input type="checkbox" id="topImpactOnly" />
    Show top-impact delta homes only (top 20%)
  </label>
</div>
"""
fmap.get_root().html.add_child(folium.Element(controls_html))

layer_ref_js = {f"{k[0]}|{k[1]}": v for k, v in layer_refs.items()}
map_name = fmap.get_name()
selector_js = f"""
<script>
(function() {{
  function byId(x) {{ return document.getElementById(x); }}
  function getMap() {{ return window['{map_name}'] || (typeof {map_name} !== 'undefined' ? {map_name} : null); }}
  var LAYERS = {layer_ref_js};

  function applyScenarioFilters() {{
    var mapObj = getMap();
    var sel = byId('scenarioSelect');
    var top = byId('topImpactOnly');
    if (!mapObj || !sel || !top) return;

    var s = sel.value;
    var showTop = top.checked;

    Object.keys(LAYERS).forEach(function(key) {{
      var layerVar = LAYERS[key];
      var lyr = window[layerVar];
      if (!lyr) return;

      var want = false;
      if (key === (s + '|baseline')) want = true;
      if (!showTop && key === (s + '|delta')) want = true;
      if (showTop && key === (s + '|delta_top20')) want = true;

      if (want) {{
        if (!mapObj.hasLayer(lyr)) mapObj.addLayer(lyr);
      }} else {{
        if (mapObj.hasLayer(lyr)) mapObj.removeLayer(lyr);
      }}
    }});
  }}

  function init() {{
    var mapObj = getMap();
    var sel = byId('scenarioSelect');
    var top = byId('topImpactOnly');
    if (!mapObj || !sel || !top) {{ setTimeout(init, 100); return; }}

    sel.value = '{scenario_order[0] if scenario_order else ''}';
    top.checked = false;
    sel.addEventListener('change', applyScenarioFilters);
    top.addEventListener('change', applyScenarioFilters);
    applyScenarioFilters();
  }}

  init();
}})();
</script>
"""
fmap.get_root().html.add_child(folium.Element(selector_js))

cmap.add_to(fmap)
fmap.save(str(MAP_OUT_HTML))
print(f'Saved map: {MAP_OUT_HTML}')
fmap

## Deliverables
- Long scenario table: `scenario_map_long.parquet` / `scenario_map_long.csv`
- Scenario summary: `scenario_summary.csv`
- Interactive comparison map: `scenario_comparison_map.html`

If you need exact annual values, rerun with:
- `USE_FAST_ANNUALIZATION = False`
- `RUN_YEARS_EXACT = 1` (or higher)
- possibly fewer areas or more compute.